# 01 · Limpieza, Carga y Validación de datos

## Objetivo

Preparar los datos contables para las siguientes fases del proyecto:

- limpiar la contabilidad;
- preparar los maestros del plan de cuentas;
- convertir los balances anuales en movimientos y saldos de cierre mensuales;
- detectar registros excluidos y diferencias entre fuentes;
- exportar tablas CSV sencillas y legibles.

> Este notebook no clasifica todavía las cuentas en masas financieras ni calcula ratios.

In [1]:
# ==========================================================
# 1. LIBRERÍAS Y RUTAS
# ==========================================================

from pathlib import Path
import re

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")


# ==========================================================
# RUTAS DEL PROYECTO
# ==========================================================

# Permite ejecutar el notebook tanto desde: la raíz del proyecto o desde la carpeta notebooks/

CWD = Path.cwd()

if (CWD / "data").exists():
    BASE_DIR = CWD
elif (CWD.parent / "data").exists():
    BASE_DIR = CWD.parent
else:
    raise FileNotFoundError(
        "No se ha podido localizar la raíz del proyecto. "
        "Debe existir una carpeta 'data/' en la raíz."
    )

RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"
OUTPUT_DIR = PROCESSED_DIR / "01"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RUTAS = {
    "contabilidad": RAW_DIR / "Conta.xlsx",
    "balances": RAW_DIR / "Balances.xlsx",
    "plan_cuentas": RAW_DIR / "Plan de cuentas.xlsx",
}


# ==========================================================
# COMPROBACIÓN DE ARCHIVOS
# ==========================================================

for nombre, ruta in RUTAS.items():
    if not ruta.exists():
        raise FileNotFoundError(
            f"No se encontró el archivo '{nombre}':\n{ruta}"
        )

print(f"Raíz del proyecto: {BASE_DIR}")
print(f"Datos raw:          {RAW_DIR}")
print(f"Salida NB1:         {OUTPUT_DIR}")
print("Archivos de entrada encontrados correctamente.")

Raíz del proyecto: C:\Users\moned\Downloads\Data_Science_Evolve\analisis_ratios
Datos raw:          C:\Users\moned\Downloads\Data_Science_Evolve\analisis_ratios\data\raw
Salida NB1:         C:\Users\moned\Downloads\Data_Science_Evolve\analisis_ratios\data\processed\01
Archivos de entrada encontrados correctamente.


In [2]:
# ==========================================================
# 2. CARGA DE ARCHIVOS
# ==========================================================

# Solo cargamos las columnas que realmente utiliza el notebook.
columnas_contabilidad = [
    "Cuenta Nivel 0",
    "Cuenta Nivel 1",
    "Cuenta Nivel 2",
    "Cuenta Nivel 5",
    "CCOSTE Nivel 0",
    "INTERCO Nivel 0",
    "Periodo contable",
    "Importe",
    "Importe debe",
    "Importe haber",
]

contabilidad_raw = pd.read_excel(
    RUTAS["contabilidad"],
    sheet_name="Conta",
    usecols=columnas_contabilidad
)

plan_n1 = pd.read_excel(
    RUTAS["plan_cuentas"],
    sheet_name="Maestro N1",
    usecols=["Cuenta N1", "Descripción N1"],
    dtype={"Cuenta N1": "string"}
)

plan_n2 = pd.read_excel(
    RUTAS["plan_cuentas"],
    sheet_name="Maestro N2",
    usecols=["Cuenta N2", "Descripción N2"],
    dtype={"Cuenta N2": "string"}
)

plan_n5 = pd.read_excel(
    RUTAS["plan_cuentas"],
    sheet_name="Maestro N5",
    usecols=["Cuenta N5", "Descripción N5"],
    dtype={"Cuenta N5": "string"}
)

print(f"Contabilidad: {len(contabilidad_raw):,} filas")
print(f"Plan N5:      {len(plan_n5):,} cuentas")

Contabilidad: 80,195 filas
Plan N5:      1,565 cuentas


In [3]:
# ==========================================================
# 3. LIMPIEZA DE LA CONTABILIDAD
# ==========================================================

# Guardamos el número de filas original antes de liberar el DataFrame raw.
filas_contabilidad_origen = len(contabilidad_raw)

contabilidad = contabilidad_raw.rename(columns={
    "Cuenta Nivel 0": "cuenta_n0",
    "Cuenta Nivel 1": "cuenta_n1",
    "Cuenta Nivel 2": "cuenta_n2",
    "Cuenta Nivel 5": "cuenta_n5",
    "CCOSTE Nivel 0": "centro_coste",
    "INTERCO Nivel 0": "intercompany",
    "Periodo contable": "periodo",
    "Importe": "importe",
    "Importe debe": "importe_debe",
    "Importe haber": "importe_haber",
})

# El DataFrame original ya no será necesario.
del contabilidad_raw


# ==========================================================
# FORMATOS BÁSICOS
# ==========================================================

for columna in ["cuenta_n0", "cuenta_n1", "cuenta_n2", "cuenta_n5"]:
    contabilidad[columna] = (
        pd.to_numeric(contabilidad[columna], errors="coerce")
        .astype("Int64")
        .astype("string")
    )

# Cuenta contable completa: siempre 6 dígitos
contabilidad["cuenta_n5"] = contabilidad["cuenta_n5"].str.zfill(6)

# Fecha
contabilidad["periodo"] = pd.to_datetime(
    contabilidad["periodo"],
    errors="coerce"
)

# Importes
for columna in ["importe", "importe_debe", "importe_haber"]:
    contabilidad[columna] = pd.to_numeric(
        contabilidad[columna],
        errors="coerce"
    )


# ==========================================================
# FILAS VÁLIDAS / EXCLUIDAS
# ==========================================================

# Controles individuales de validez
periodo_valido = contabilidad["periodo"].notna()

cuenta_valida = contabilidad["cuenta_n5"].str.fullmatch(
    r"\d{6}",
    na=False
)

importe_valido = contabilidad["importe"].notna()

# Una fila es válida si cumple los tres criterios
es_valida = (
    periodo_valido
    & cuenta_valida
    & importe_valido
)

contabilidad_excluida = contabilidad.loc[~es_valida].copy()
contabilidad_limpia = contabilidad.loc[es_valida].copy()


# ==========================================================
# MOTIVO DE EXCLUSIÓN
# ==========================================================

contabilidad_excluida["motivo_exclusion"] = ""

contabilidad_excluida.loc[
    ~periodo_valido.loc[contabilidad_excluida.index],
    "motivo_exclusion"
] += "Periodo inválido; "

contabilidad_excluida.loc[
    ~cuenta_valida.loc[contabilidad_excluida.index],
    "motivo_exclusion"
] += "Cuenta N5 inválida; "

contabilidad_excluida.loc[
    ~importe_valido.loc[contabilidad_excluida.index],
    "motivo_exclusion"
] += "Importe inválido; "

contabilidad_excluida["motivo_exclusion"] = (
    contabilidad_excluida["motivo_exclusion"]
    .str.rstrip("; ")
)


# ==========================================================
# RESUMEN DE EXCLUSIONES
# ==========================================================

resumen_exclusiones = (
    contabilidad_excluida["motivo_exclusion"]
    .value_counts()
    .rename_axis("motivo")
    .reset_index(name="filas")
)

print(f"Filas válidas:   {len(contabilidad_limpia):,}")
print(f"Filas excluidas: {len(contabilidad_excluida):,}")

display(resumen_exclusiones)


# Ya no necesitamos las variables auxiliares.
del contabilidad
del es_valida
del periodo_valido
del cuenta_valida
del importe_valido


# ==========================================================
# COLUMNAS PARA ANÁLISIS Y POWER BI
# ==========================================================

contabilidad_limpia["centro_coste"] = (
    contabilidad_limpia["centro_coste"]
    .astype("string")
    .fillna("SIN CENTRO")
    .str.strip()
)

contabilidad_limpia["intercompany"] = (
    contabilidad_limpia["intercompany"]
    .astype("string")
    .fillna("NO")
    .str.strip()
)

contabilidad_limpia["año"] = (
    contabilidad_limpia["periodo"]
    .dt.year
    .astype("int16")
)

contabilidad_limpia["mes_numero"] = (
    contabilidad_limpia["periodo"]
    .dt.month
    .astype("int8")
)

contabilidad_limpia["año_mes"] = (
    contabilidad_limpia["periodo"]
    .dt.strftime("%Y-%m")
)

contabilidad_limpia["trimestre"] = (
    "T"
    + contabilidad_limpia["periodo"]
    .dt.quarter
    .astype(str)
)


# ==========================================================
# ORDEN FINAL
# ==========================================================

columnas_finales = [
    "periodo",
    "año",
    "mes_numero",
    "año_mes",
    "trimestre",
    "cuenta_n0",
    "cuenta_n1",
    "cuenta_n2",
    "cuenta_n5",
    "centro_coste",
    "intercompany",
    "importe",
    "importe_debe",
    "importe_haber",
]

contabilidad_limpia = contabilidad_limpia[columnas_finales]

display(contabilidad_limpia.head())

Filas válidas:   78,688
Filas excluidas: 1,507


,motivo,filas
0,Periodo inválido; Importe inválido,1362
1,Periodo inválido; Cuenta N5 inválida; Importe ...,96
2,Cuenta N5 inválida; Importe inválido,49


,periodo,año,mes_numero,año_mes,trimestre,cuenta_n0,cuenta_n1,cuenta_n2,cuenta_n5,centro_coste,intercompany,importe,importe_debe,importe_haber
0,2021-01-01,2021,1,2021-01,T1,2,21,213,213000,S002,NO,0.00,0.00,0.00
1,2021-01-01,2021,1,2021-01,T1,2,21,213,213000,S002,i1,"-96,800.00",0.00,"96,800.00"
2,2021-01-01,2021,1,2021-01,T1,2,21,213,213000,S002,i2,"-7,621.38",0.00,"7,621.00"
3,2021-01-01,2021,1,2021-01,T1,2,21,213,213000,S003,NO,"6,171.00","6,171.00",0.00
4,2021-02-01,2021,2,2021-02,T1,2,21,213,213000,S003,NO,622.04,622.00,0.00


In [4]:
# ==========================================================
# 4. LIMPIEZA DEL PLAN DE CUENTAS
# ==========================================================

def limpiar_maestro(df, cuenta, descripcion, longitud):
    salida = df[[cuenta, descripcion]].copy()
    salida[cuenta] = (
        pd.to_numeric(salida[cuenta], errors="coerce")
        .astype("Int64")
        .astype("string")
        .str.zfill(longitud)
    )
    salida[descripcion] = salida[descripcion].astype("string").str.strip()
    return salida.dropna(subset=[cuenta]).drop_duplicates(subset=[cuenta])

plan_n1 = limpiar_maestro(plan_n1, "Cuenta N1", "Descripción N1", 1).rename(columns={
    "Cuenta N1": "cuenta_n1",
    "Descripción N1": "descripcion_n1"
})

plan_n2 = limpiar_maestro(plan_n2, "Cuenta N2", "Descripción N2", 2).rename(columns={
    "Cuenta N2": "cuenta_n2",
    "Descripción N2": "descripcion_n2"
})

plan_n5 = limpiar_maestro(plan_n5, "Cuenta N5", "Descripción N5", 6).rename(columns={
    "Cuenta N5": "cuenta_n5",
    "Descripción N5": "descripcion_n5"
})

cuentas_contabilidad = set(contabilidad_limpia["cuenta_n5"].dropna())
cuentas_maestro = set(plan_n5["cuenta_n5"].dropna())
cuentas_sin_maestro = sorted(cuentas_contabilidad - cuentas_maestro)

print(f"Cuentas utilizadas:          {len(cuentas_contabilidad):,}")
print(f"Cuentas sin maestro N5:      {len(cuentas_sin_maestro):,}")

Cuentas utilizadas:          240
Cuentas sin maestro N5:      0


In [5]:
# ==========================================================
# 5. BALANCES: MOVIMIENTOS Y SALDOS DE CIERRE
# ==========================================================

def fecha_columna(texto):
    """
    Extrae la última fecha DD/MM/YY encontrada
    en el nombre de una columna del balance.
    """
    fechas = re.findall(r"\d{2}/\d{2}/\d{2}", str(texto))

    if not fechas:
        return pd.NaT

    return pd.to_datetime(
        fechas[-1],
        dayfirst=True,
        errors="coerce"
    )


movimientos_balance = []
saldos_balance = []


# Abrimos el Excel una sola vez.
with pd.ExcelFile(RUTAS["balances"]) as libro_balances:

    for hoja in libro_balances.sheet_names:

        balance = pd.read_excel(
            libro_balances,
            sheet_name=hoja
        )

        # --------------------------------------------------
        # CUENTA Y DESCRIPCIÓN
        # --------------------------------------------------

        # Primera columna:
        # código de cuenta + descripción.
        detalle = (
            balance.iloc[:, 0]
            .astype("string")
            .str.extract(r"^\s*(\d+)\s+(.*)$")
        )

        balance["cuenta_n5"] = detalle[0]
        balance["descripcion_n5"] = detalle[1].str.strip()

        # Solo cuentas contables de 6 dígitos.
        # Así evitamos incorporar subtotales del balance.
        balance = balance.loc[
            balance["cuenta_n5"].str.fullmatch(r"\d{6}", na=False)
        ].copy()

        if balance.empty:
            continue


        # --------------------------------------------------
        # SALDO INICIAL
        # --------------------------------------------------

        columna_inicial = balance.columns[1]

        saldo_acumulado = (
            pd.to_numeric(
                balance[columna_inicial],
                errors="coerce"
            )
            .fillna(0.0)
        )


        # --------------------------------------------------
        # MOVIMIENTOS MENSUALES
        # --------------------------------------------------

        for columna in balance.columns[2:]:

            fecha = fecha_columna(columna)

            if pd.isna(fecha):
                continue

            periodo = fecha.to_period("M").to_timestamp()

            movimiento_mes = (
                pd.to_numeric(
                    balance[columna],
                    errors="coerce"
                )
                .fillna(0.0)
            )

            saldo_acumulado = saldo_acumulado + movimiento_mes

            base = pd.DataFrame({
                "periodo": periodo,
                "cuenta_n5": balance["cuenta_n5"].to_numpy(),
                "descripcion_n5": balance["descripcion_n5"].to_numpy(),
            })

            movimientos_balance.append(
                base.assign(
                    movimiento_mes=movimiento_mes.to_numpy()
                )
            )

            saldos_balance.append(
                base.assign(
                    saldo_cierre=saldo_acumulado.to_numpy()
                )
            )


# ==========================================================
# UNIÓN FINAL
# ==========================================================

if not movimientos_balance or not saldos_balance:
    raise ValueError(
        "No se han detectado movimientos mensuales válidos "
        "en el archivo de balances."
    )

balance_movimientos = pd.concat(
    movimientos_balance,
    ignore_index=True
)

balance_saldos_cierre = pd.concat(
    saldos_balance,
    ignore_index=True
)

# Liberamos las listas de DataFrames intermedios.
del movimientos_balance
del saldos_balance


# ==========================================================
# COLUMNAS TEMPORALES
# ==========================================================

for df in [balance_movimientos, balance_saldos_cierre]:

    df["año"] = df["periodo"].dt.year.astype("int16")
    df["mes_numero"] = df["periodo"].dt.month.astype("int8")
    df["año_mes"] = df["periodo"].dt.strftime("%Y-%m")


# ==========================================================
# CONTROL
# ==========================================================

print(
    f"Meses disponibles: "
    f"{balance_saldos_cierre['periodo'].nunique()}"
)

print(
    f"Periodo: "
    f"{balance_saldos_cierre['periodo'].min():%Y-%m} - "
    f"{balance_saldos_cierre['periodo'].max():%Y-%m}"
)

display(balance_saldos_cierre.head())

Meses disponibles: 36
Periodo: 2021-01 - 2023-12


,periodo,cuenta_n5,descripcion_n5,saldo_cierre,año,mes_numero,año_mes
0,2021-01-01,100000,Capital Social,"-3,000.00",2021,1,2021-01
1,2021-01-01,100001,Capital Social por ampliaciones de Capital,"-51,000.00",2021,1,2021-01
2,2021-01-01,103000,Socios por desembolsos no exigidos,0.00,2021,1,2021-01
3,2021-01-01,110000,Prima Emisión,"-4,949,000.00",2021,1,2021-01
4,2021-01-01,112000,Reserva legal,0.00,2021,1,2021-01


In [6]:
# ==========================================================
# 6. CONTROL ENTRE CONTABILIDAD Y BALANCE
# ==========================================================

# ==========================================================
# MOVIMIENTO SEGÚN CONTABILIDAD
# ==========================================================

mov_contabilidad = (
    contabilidad_limpia
    .groupby(
        ["periodo", "cuenta_n5"],
        as_index=False,
        observed=True
    )["importe"]
    .sum()
    .rename(columns={
        "importe": "movimiento_contabilidad"
    })
)


# ==========================================================
# MOVIMIENTO SEGÚN BALANCE
# ==========================================================

mov_balance_control = (
    balance_movimientos
    .groupby(
        ["periodo", "cuenta_n5"],
        as_index=False,
        observed=True
    )
    .agg(
        descripcion_n5=("descripcion_n5", "first"),
        movimiento_balance=("movimiento_mes", "sum")
    )
)


# ==========================================================
# COMPARACIÓN
# ==========================================================

control_balance = mov_balance_control.merge(
    mov_contabilidad,
    on=["periodo", "cuenta_n5"],
    how="outer",
    validate="one_to_one"
)

control_balance["movimiento_balance"] = (
    control_balance["movimiento_balance"]
    .fillna(0.0)
)

control_balance["movimiento_contabilidad"] = (
    control_balance["movimiento_contabilidad"]
    .fillna(0.0)
)

control_balance["diferencia"] = (
    control_balance["movimiento_balance"]
    - control_balance["movimiento_contabilidad"]
)

control_balance["estado"] = np.where(
    control_balance["diferencia"].abs() <= 0.01,
    "CUADRA",
    "REVISAR"
)


# ==========================================================
# ORDEN FINAL
# ==========================================================

control_balance = control_balance[[
    "periodo",
    "cuenta_n5",
    "descripcion_n5",
    "movimiento_balance",
    "movimiento_contabilidad",
    "diferencia",
    "estado",
]]

control_balance = control_balance.sort_values(
    ["periodo", "cuenta_n5"]
).reset_index(drop=True)


print(control_balance["estado"].value_counts())

display(
    control_balance.loc[
        control_balance["estado"] == "REVISAR"
    ]
    .sort_values(
        "diferencia",
        key=lambda x: x.abs(),
        ascending=False
    )
    .head(10)
)


# ==========================================================
# RESUMEN DE CONCILIACIÓN CONTABILIDAD VS BALANCE
# ==========================================================

# Control a nivel cuenta-mes
control_balance["estado_cuenta"] = np.where(
    control_balance["diferencia"].abs() <= 0.01,
    "CUADRA",
    "DIFERENCIA DE CLASIFICACIÓN"
)

control_mensual = (
    control_balance
    .groupby("periodo", as_index=False)
    .agg(
        diferencia_neta=("diferencia", "sum"),
        diferencia_absoluta=("diferencia", lambda x: x.abs().sum())
    )
)

control_mensual["estado_mes"] = np.where(
    control_mensual["diferencia_neta"].abs() <= 0.01,
    "CUADRA",
    "REVISAR"
)


meses_totales = control_mensual["periodo"].nunique()

meses_cuadrados = (
    control_mensual["estado_mes"] == "CUADRA"
).sum()

diferencias_cuenta_mes = (
    control_balance["estado_cuenta"] == "DIFERENCIA DE CLASIFICACIÓN"
).sum()

diferencia_neta_total = control_balance["diferencia"].sum()

diferencia_absoluta_total = control_balance["diferencia"].abs().sum()

print("CONTROL CONTABILIDAD VS BALANCE")
print("-" * 40)

print(
    f"Meses conciliados: "
    f"{meses_cuadrados}/{meses_totales}"
)

print(
    f"Diferencias cuenta-mes detectadas: "
    f"{diferencias_cuenta_mes:,}"
)

print(
    f"Diferencia neta total: "
    f"{diferencia_neta_total:,.2f} €"
)

print(
    f"Diferencia absoluta entre cuentas: "
    f"{diferencia_absoluta_total:,.2f} €"
)

if meses_cuadrados == meses_totales:
    print(
        "\nResultado: todos los periodos cuadran a nivel agregado."
    )
    print(
        "Las diferencias detectadas corresponden a redistribuciones "
        "entre cuentas dentro del mismo periodo."
    )
else:
    print(
        "\nAVISO: existen periodos con diferencias netas entre "
        "Contabilidad y Balance."
    )


estado
CUADRA     54610
REVISAR     1730
Name: count, dtype: int64


,periodo,cuenta_n5,descripcion_n5,movimiento_balance,movimiento_contabilidad,diferencia,estado
18793,2022-01-01,121000,Resultados negativos de ejer.anter.,0.00,"134,349.67","-134,349.67",REVISAR
36006,2022-12-01,118000,Otras aportaciones de socios,0.00,"-130,000.00","130,000.00",REVISAR
43834,2023-05-01,129000,Pérdidas y ganancias,"847,234.02","727,591.43","119,642.59",REVISAR
43833,2023-05-01,121000,Resultados negativos de ejer.anter.,"-836,434.02","-716,791.43","-119,642.59",REVISAR
32925,2022-10-01,213000,Maquinaria (slots machines),"9,927.42","113,442.92","-103,515.50",REVISAR
18791,2022-01-01,118000,Otras aportaciones de socios,0.00,"-98,816.00","98,816.00",REVISAR
18780,2022-01-01,100000,Capital Social,0.00,"-90,151.82","90,151.82",REVISAR
33098,2022-10-01,410000,"Acreedores, Nacional","-23,552.19","-108,413.91","84,861.72",REVISAR
18840,2022-01-01,213000,Maquinaria (slots machines),65.30,"61,872.18","-61,806.88",REVISAR
36414,2022-12-01,552000,Cta.Corriente empresas GRUPO-Pasivo,"24,859.73","77,246.31","-52,386.58",REVISAR


CONTROL CONTABILIDAD VS BALANCE
----------------------------------------
Meses conciliados: 36/36
Diferencias cuenta-mes detectadas: 1,730
Diferencia neta total: 0.00 €
Diferencia absoluta entre cuentas: 4,507,413.68 €

Resultado: todos los periodos cuadran a nivel agregado.
Las diferencias detectadas corresponden a redistribuciones entre cuentas dentro del mismo periodo.


In [7]:
# ==========================================================
# 7. INFORME DE CALIDAD
# ==========================================================

# Conciliación mensual entre Contabilidad y Balance
control_mensual = (
    control_balance
    .groupby("periodo", as_index=False)
    .agg(
        cuentas_con_diferencia=(
            "estado",
            lambda x: (x == "REVISAR").sum()
        ),
        diferencia_neta=("diferencia", "sum"),
        diferencia_absoluta=(
            "diferencia",
            lambda x: x.abs().sum()
        ),
    )
)

control_mensual["estado"] = np.where(
    control_mensual["diferencia_neta"].abs() <= 0.01,
    "CUADRA",
    "REVISAR"
)

meses_totales = len(control_mensual)

meses_conciliados = (
    control_mensual["estado"] == "CUADRA"
).sum()

diferencias_cuenta_mes = (
    control_balance["estado"] == "REVISAR"
).sum()

diferencia_neta_maxima = (
    control_mensual["diferencia_neta"]
    .abs()
    .max()
)


# ==========================================================
# RESUMEN
# ==========================================================

informe_calidad = pd.DataFrame([
    [
        "filas_contabilidad_origen",
        filas_contabilidad_origen
    ],
    [
        "filas_contabilidad_validas",
        len(contabilidad_limpia)
    ],
    [
        "filas_contabilidad_excluidas",
        len(contabilidad_excluida)
    ],
    [
        "cuentas_n5_utilizadas",
        contabilidad_limpia["cuenta_n5"].nunique()
    ],
    [
        "cuentas_n5_sin_maestro",
        len(cuentas_sin_maestro)
    ],
    [
        "centros_coste",
        contabilidad_limpia["centro_coste"].nunique()
    ],
    [
        "filas_sin_centro_coste",
        (
            contabilidad_limpia["centro_coste"]
            == "SIN CENTRO"
        ).sum()
    ],
    [
        "periodo_inicial",
        contabilidad_limpia["periodo"]
        .min()
        .strftime("%Y-%m-%d")
    ],
    [
        "periodo_final",
        contabilidad_limpia["periodo"]
        .max()
        .strftime("%Y-%m-%d")
    ],
    [
        "meses_conciliados_contabilidad_balance",
        f"{meses_conciliados}/{meses_totales}"
    ],
    [
        "diferencias_clasificacion_cuenta_mes",
        diferencias_cuenta_mes
    ],
    [
        "diferencia_neta_mensual_maxima",
        round(diferencia_neta_maxima, 2)
    ],
], columns=["indicador", "valor"])


print("INFORME DE CALIDAD")
print("-" * 50)

display(informe_calidad)

print(
    f"\nConciliación Contabilidad-Balance: "
    f"{meses_conciliados}/{meses_totales} meses."
)

print(
    f"Diferencias de clasificación cuenta-mes: "
    f"{diferencias_cuenta_mes:,}."
)

print(
    f"Diferencia neta mensual máxima: "
    f"{diferencia_neta_maxima:,.2f} €."
)

if meses_conciliados == meses_totales:
    print(
        "Resultado: todos los meses cuadran a nivel agregado; "
        "las diferencias entre cuentas se compensan dentro de cada periodo."
    )
else:
    print(
        "AVISO: existen meses con diferencias netas entre "
        "Contabilidad y Balance."
    )

INFORME DE CALIDAD
--------------------------------------------------


,indicador,valor
0,filas_contabilidad_origen,80195
1,filas_contabilidad_validas,78688
2,filas_contabilidad_excluidas,1507
3,cuentas_n5_utilizadas,240
4,cuentas_n5_sin_maestro,0
5,centros_coste,30
6,filas_sin_centro_coste,2272
7,periodo_inicial,2021-01-01
8,periodo_final,2023-12-01
9,meses_conciliados_contabilidad_balance,36/36



Conciliación Contabilidad-Balance: 36/36 meses.
Diferencias de clasificación cuenta-mes: 1,730.
Diferencia neta mensual máxima: 0.00 €.
Resultado: todos los meses cuadran a nivel agregado; las diferencias entre cuentas se compensan dentro de cada periodo.


In [8]:
# ==========================================================
# 8. EXPORTACIÓN
# ==========================================================

EXPORTACIONES = {
    "01_contabilidad_limpia.csv": contabilidad_limpia,
    "01_contabilidad_excluida.csv": contabilidad_excluida,
    "01_plan_cuentas_n1.csv": plan_n1,
    "01_plan_cuentas_n2.csv": plan_n2,
    "01_plan_cuentas_n5.csv": plan_n5,
    "01_balance_movimientos.csv": balance_movimientos,
    "01_balance_saldos_cierre.csv": balance_saldos_cierre,
    "01_control_contabilidad_balance.csv": control_balance,
    "01_informe_calidad.csv": informe_calidad,
}


for nombre, df in EXPORTACIONES.items():

    ruta_salida = OUTPUT_DIR / nombre

    df.to_csv(
        ruta_salida,
        index=False,
        encoding="utf-8-sig",
        decimal="."
    )


print("Archivos exportados correctamente:")

for nombre in EXPORTACIONES:
    print(f" - data/processed/01/{nombre}")

Archivos exportados correctamente:
 - data/processed/01/01_contabilidad_limpia.csv
 - data/processed/01/01_contabilidad_excluida.csv
 - data/processed/01/01_plan_cuentas_n1.csv
 - data/processed/01/01_plan_cuentas_n2.csv
 - data/processed/01/01_plan_cuentas_n5.csv
 - data/processed/01/01_balance_movimientos.csv
 - data/processed/01/01_balance_saldos_cierre.csv
 - data/processed/01/01_control_contabilidad_balance.csv
 - data/processed/01/01_informe_calidad.csv


## Salidas del notebook

Las tablas principales para continuar el proyecto son:

- `01_contabilidad_limpia.csv`: movimientos contables válidos y preparados para Power BI.
- `01_plan_cuentas_n5.csv`: catálogo de cuentas.
- `01_balance_movimientos.csv`: movimientos mensuales procedentes del balance.
- `01_balance_saldos_cierre.csv`: saldo de cada cuenta al cierre de cada mes.
- `01_control_contabilidad_balance.csv`: diferencias entre ambas fuentes.
- `01_informe_calidad.csv`: resumen de calidad y cobertura.

Las filas de nivel 1, 2 y 3 del archivo de balances no se incorporan a los cálculos para evitar duplicar importes.